### Cleaning Data

The general approach to finding dirty data: you don't just look at the data and eyeball it, you run a specific set of diagnostic checks, each one designed to catch a different category of problem. Here's the order that makes sense:

1. Basic shape and structure (rows, columns, dtypes)
2. Missing values
3. Duplicate rows
4. Categorical consistency (are labels spelled consistently?)
5. Numeric plausibility (are values within realistic ranges?)
6. Type contamination (are "numbers" secretly stored as text?)

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/synthetic_sickle_cell_data_MESSY.csv")

#### STEP 1: Basic shape and structure (rows, columns, dtypes)

In [5]:
print(df.shape)
print(df.dtypes)
df.head(5)

(2130, 14)
age                   float64
genotype                  str
baseline_hb_g_dl          str
hbf_percent           float64
wbc_10e9_l                str
reticulocyte_pct      float64
prior_voc_6mo         float64
crp_mg_l              float64
temperature_c         float64
hydration_score       float64
spo2_pct              float64
pain_duration_days    float64
season                    str
cause_label               str
dtype: object


,age,genotype,baseline_hb_g_dl,hbf_percent,wbc_10e9_l,reticulocyte_pct,prior_voc_6mo,crp_mg_l,temperature_c,hydration_score,spo2_pct,pain_duration_days,season,cause_label
0,37.5,SS,6.02,5.18,20.05,11.58,2.0,NaN,38.46,NaN,95.14,1.90,rainy,ACS
1,20.4,SBeta-thal,9.35,1.17,9.24,NaN,0.0,8.97,36.68,3.29,98.19,37.94,rainy,AVN
2,18.0,SS,8.22,8.41,13.37,7.53,6.0,19.35,36.91,1.26,99.79,2.82,dry,VOC
3,29.5,SS,8.06,0.00,12.32,6.75,1.0,6.26,37.71,2.25,97.25,1.36,dry season,Dehydration
4,26.7,SS,7.84,11.55,14.54,12.81,1.0,22.50,36.38,3.61,98.06,8.99,rainy,Other


In [6]:
# is age genuinely decimal, or just float-typed because of a missing value somewhere?
print(df["age"].isna().sum())  # any missing values?
print((df["age"] % 1 == 0).all())  # True if every value is a whole number

5
False


In [7]:
# find which data parses as numeric
def is_clean_number(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False


bad_hb = df[~df["baseline_hb_g_dl"].apply(is_clean_number)]
print(bad_hb[["baseline_hb_g_dl"]].head(10))
print(f"\n{len(bad_hb)} contaminated rows out of {len(df)}")

    baseline_hb_g_dl
23          7.1 g/dl
46         9.36 g/dl
58         7.46 g/dl
80         8.26 g/dl
98         8.26 g/dl
139        9.14 g/dl
144         6.8 g/dl
159        7.71 g/dl
188        9.88 g/dl
203        7.66 g/dl

106 contaminated rows out of 2130


In [8]:
import pandas as pd

n_df = pd.DataFrame({
    "id": [1, 2, 3],
    "age": [24, 31, 45]
})

n_df[[True, False, False]]

,id,age
0,1,24


In [9]:
bad_wbc = df[~df["wbc_10e9_l"].apply(is_clean_number)]
print(bad_wbc[["wbc_10e9_l"]].head(10))
print(f"\n{len(bad_wbc)} contaminated rows out of {len(df)}")

        wbc_10e9_l
34   11.87 x10^9/L
52   10.23 x10^9/L
57   19.96 x10^9/L
85   17.89 x10^9/L
120  10.47 x10^9/L
123  15.74 x10^9/L
139   7.27 x10^9/L
143    5.0 x10^9/L
152  12.34 x10^9/L
174  16.71 x10^9/L

106 contaminated rows out of 2130


In [10]:
print(not True)
print(df["baseline_hb_g_dl"].apply(is_clean_number))
# Convert False to True and vice versa
print(~df["baseline_hb_g_dl"].apply(is_clean_number))

False
0        True
1        True
2        True
3        True
4        True
        ...  
2125     True
2126     True
2127    False
2128     True
2129     True
Name: baseline_hb_g_dl, Length: 2130, dtype: bool
0       False
1       False
2       False
3       False
4       False
        ...  
2125    False
2126    False
2127     True
2128    False
2129    False
Name: baseline_hb_g_dl, Length: 2130, dtype: bool


### Step 2, 4, 5, and 6

2. Missing values
4. Categorical consistency (are labels spelled consistently?)
5. Numeric plausibility (are values within realistic ranges?)
6. Type contamination (are "numbers" secretly stored as text?)

In [11]:
# Mini Profiling
for col in df.columns:
    print(f"\n--- {col} ---")
    print(f"dtype: {df[col].dtype}")
    print(f"missing: {df[col].isna().sum()} ({df[col].isna().sum() / len(df) * 100:.1f}%)")
    print(f"unique values: {df[col].nunique()}")
    if df[col].dtype in ["int64", "float64"]:
        print(f"min: {df[col].min()}, max: {df[col].max()}")
    else:
        print(f"sample values: {df[col].dropna().unique()[:8]}")


--- age ---
dtype: float64
missing: 5 (0.2%)
unique values: 449
min: -5.0, max: 65.0

--- genotype ---
dtype: str
missing: 5 (0.2%)
unique values: 16
sample values: <StringArray>
['SS', 'SBeta-thal', 'SC', 'Hb SS', 'Hb-SC', 'Hb-SS', 'S/beta-thal', ' SS ']
Length: 8, dtype: str

--- baseline_hb_g_dl ---
dtype: str
missing: 5 (0.2%)
unique values: 551
sample values: <StringArray>
['6.02', '9.35', '8.22', '8.06', '7.84', '6.09', '8.98', '7.32']
Length: 8, dtype: str

--- hbf_percent ---
dtype: float64
missing: 91 (4.3%)
unique values: 930
min: 0.0, max: 16.31

--- wbc_10e9_l ---
dtype: str
missing: 5 (0.2%)
unique values: 1251
sample values: <StringArray>
['20.05', '9.24', '13.37', '12.32', '14.54', '9.9', '12.27', '20.29']
Length: 8, dtype: str

--- reticulocyte_pct ---
dtype: float64
missing: 91 (4.3%)
unique values: 868
min: 2.0, max: 17.21

--- prior_voc_6mo ---
dtype: float64
missing: 5 (0.2%)
unique values: 13
min: 0.0, max: 13.0

--- crp_mg_l ---
dtype: float64
missing: 91 (4.3%)


### Step 3: Duplicate rows

In [12]:
print(df.duplicated().sum())

29


### Make a copy to apply Fixes 

In [48]:
import numpy as np

df_clean = df.copy()
df_clean.shape

(2130, 14)

Three modes of dropna, all doing genuinely different things:

- dropna() (default, how="any"): drop if any column missing, most aggressive
- dropna(how="all"): drop only if every column missing, least aggressive
- dropna(subset=[...]): drop only if specific named column(s) missing, targeted

In [49]:
# Step 1: Drop fully empty rows ---
df_clean = df_clean.dropna(how="all").reset_index(drop=True)
df_clean.shape

(2125, 14)

In [50]:
# Step 2: Drop exact duplicate rows ---
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
df_clean.shape

(2100, 14)

In [51]:
df_clean.iloc[2097]["baseline_hb_g_dl"]

'7.93 g/dl'

In [52]:
# Step 3: Fix type contamination: strip units, convert to numeric
for col in ["baseline_hb_g_dl", "wbc_10e9_l"]:
    df_clean[col] = df_clean[col].astype(object)  # allow mixed content before conversion
    df_clean[col] = df_clean[col].astype(str).str.extract(r"(-?\d+\.?\d*)")[0]
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
    
df_clean.iloc[2097]["baseline_hb_g_dl"]

np.float64(7.93)

In [ ]:
# Step 4: normalize genotype spelling

genotype_map = {}

for val in df_clean["genotype"].dropna().unique():
    v = val.strip().lower().replace("hb-", "").replace("hb ", "").replace("hb", "").replace("/", "").replace(" ", "")
    if v == "ss":
        genotype_map[val] = "SS"
    elif v == "sc":
        genotype_map[val] = "SC"
    elif "beta" in v:
        genotype_map[val] = "SBeta-thal"

df_clean["genotype"] = df_clean["genotype"].map(genotype_map)

In [54]:
print("== NEW ==")
print(df_clean["genotype"].unique())
print("\n== OLD ==")
print(df["genotype"].unique())

== NEW ==
<StringArray>
['SS', 'SBeta-thal', 'SC']
Length: 3, dtype: str

== OLD ==
<StringArray>
[         'SS',  'SBeta-thal',          'SC',       'Hb SS',       'Hb-SC',
       'Hb-SS', 'S/beta-thal',        ' SS ',        ' SC ',          'ss',
        'HBSS',        'HBSC',          'sc',           nan,  'sbeta-thal',
  'Sbeta thal', 'S-beta thal']
Length: 17, dtype: str


In [55]:
print(df_clean["genotype"].isna().sum())

0


In [56]:
print(df["genotype"].isna().sum())

5


In [57]:
# Step 5: normalize season spelling

season_map = {}

for val in df_clean["season"].dropna().unique():
    v = val.strip().lower().replace(" season", "")
    if v in ["dry", "rainy"]:
        season_map[val] = v

df_clean["season"] = df_clean["season"].map(season_map)

print(df_clean["season"].unique())



<StringArray>
['rainy', 'dry']
Length: 2, dtype: str


In [63]:
# Step 6: null out impossible numeric values

import numpy as np

df_clean.loc[df_clean["age"] < 0, "age"] = np.nan
df_clean.loc[(df_clean["temperature_c"] < 34) | (df_clean["temperature_c"] > 42), "temperature_c"] = np.nan
df_clean.loc[(df_clean["spo2_pct"] < 50) | (df_clean["spo2_pct"] > 100), "spo2_pct"] = np.nan

print(df_clean[["age", "temperature_c", "spo2_pct"]].isna().sum())


age              3
temperature_c    2
spo2_pct         7
dtype: int64


In [64]:
# Step 7: drop rows missing the target label

df_clean = df_clean.dropna(subset=["cause_label"]).reset_index(drop=True)
print(f"Rows remaining: {len(df_clean)}")

Rows remaining: 2100


In [65]:
# Step 8: impute remaining missing values

numeric_cols = ["age", "baseline_hb_g_dl", "hbf_percent", "wbc_10e9_l", "reticulocyte_pct",
                "prior_voc_6mo", "crp_mg_l", "temperature_c", "hydration_score",
                "spo2_pct", "pain_duration_days"]

for col in numeric_cols:
    df_clean[col] = df_clean.groupby("cause_label")[col].transform(lambda s: s.fillna(s.median()))

for col in ["genotype", "season"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"Final shape: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isna().sum().sum()}")
print(f"Remaining duplicates: {df_clean.duplicated().sum()}")

Final shape: (2100, 14)
Remaining missing values: 0
Remaining duplicates: 0


In [66]:
# save the cleaned dataset

df_clean.to_csv("../data/processed/synthetic_sickle_cell_data_CLEANED.csv", index=False)